In [2]:
import pandas as pd
import numpy as np
import altair as alt
import requests as rqs

### Certificado HTTPS do IBGE

O servidor omite o certificado intermediario Sectigo Public Server Authentication CA DV R36.
As duas consultas usam `criar_sessao_ibge()`, que complementa a cadeia com
`certs/SectigoPublicServerAuthenticationCADVR36.pem` e valida o dominio, a validade e a raiz confiavel.
A configuracao se aplica somente a `https://servicodados.ibge.gov.br/`.
Execute este arquivo `API_IBGE.ipynb` com a pasta do projeto como diretorio de trabalho.

Se o notebook ja estava aberto ao receber alteracoes externas, recarregue-o do disco
antes de salvar, para evitar sobrescrever estas celulas com uma copia antiga do navegador.
Depois, reinicie o kernel `AED082026` e execute todas as celulas em ordem.

Fonte: http://crt.sectigo.com/SectigoPublicServerAuthenticationCADVR36.crt

SHA-256 do certificado DER: `8c54c334b66ba4e426772af4a3f9136c19a1aec729fdb28c535c07a5a4ef22e0`.
Cadeia validada em 14/10/2026. Revise o intermediario se o IBGE trocar de emissor.


In [ ]:
import ssl
from pathlib import Path
from requests.adapters import HTTPAdapter


class IBGETLSAdapter(HTTPAdapter):
    """Completa a cadeia do IBGE sem desativar a validacao TLS."""

    def __init__(self, certificado_intermediario, **kwargs):
        self.ssl_context = ssl.create_default_context(cafile=rqs.certs.where())
        # O intermediario deve encadear ate uma raiz confiavel, sem cadeia parcial.
        self.ssl_context.verify_flags &= ~ssl.VERIFY_X509_PARTIAL_CHAIN
        self.ssl_context.load_verify_locations(cafile=str(certificado_intermediario))
        super().__init__(**kwargs)

    def init_poolmanager(self, connections, maxsize, block=False, **pool_kwargs):
        pool_kwargs['ssl_context'] = self.ssl_context
        return super().init_poolmanager(connections, maxsize, block=block, **pool_kwargs)

    def proxy_manager_for(self, proxy, **proxy_kwargs):
        proxy_kwargs['ssl_context'] = self.ssl_context
        return super().proxy_manager_for(proxy, **proxy_kwargs)


def criar_sessao_ibge():
    certificado = Path("certs/SectigoPublicServerAuthenticationCADVR36.pem")
    adapter = IBGETLSAdapter(certificado)
    sessao = rqs.Session()
    sessao.mount("https://servicodados.ibge.gov.br/", adapter)
    return sessao


In [ ]:
tabela_lavouras_temporárias = 1612
url_metadados = f"https://servicodados.ibge.gov.br/api/v3/agregados/{tabela_lavouras_temporárias}/metadados"

with criar_sessao_ibge() as sessao:
    response = sessao.get(url_metadados, timeout=30)
    response.raise_for_status()
    metadados = response.json()


In [ ]:
print("Tabela:", metadados["nome"])

for variavel in metadados["variaveis"]:
    print(
        variavel["id"],
        variavel["nome"],
        variavel["unidade"]
    )

Tabela: Área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias
109 Área plantada Hectares
1000109 Área plantada - percentual do total geral %
216 Área colhida Hectares
1000216 Área colhida - percentual do total geral %
214 Quantidade produzida Toneladas
112 Rendimento médio da produção Quilogramas por Hectare
215 Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992], Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
1000215 Valor da produção - percentual do total geral %


In [ ]:
for classificacao in metadados["classificacoes"]:
    print("Classificação:", classificacao["id"], classificacao["nome"])

    for categoria in classificacao["categorias"]:
        print(categoria["id"], categoria["nome"])

Classificação: 81 Produto das lavouras temporárias
0 Total
2688 Abacaxi*
83374 Abóbora
40471 Alfafa fenada
83375 Alface
2689 Algodão herbáceo (em caroço)
2690 Alho
2691 Amendoim (em casca)
2692 Arroz (em casca)
2693 Aveia (em grão)
2694 Batata-doce
2695 Batata-inglesa
2696 Cana-de-açúcar
40470 Cana para forragem
83384 Canola
2697 Cebola
83381 Cenoura
2698 Centeio (em grão)
2699 Cevada (em grão)
83379 Chuchu
2700 Ervilha (em grão)
2701 Fava (em grão)
2702 Feijão (em grão)
2703 Fumo (em folha)
83383 Gergelim
109179 Girassol (em grão)
83382 Inhame
2704 Juta (fibra)
2705 Linho (semente)
2706 Malva (fibra)
2707 Mamona (baga)
2708 Mandioca
2709 Melancia
2710 Melão
2711 Milho (em grão)
83377 Milho verde
83378 Morango
83376 Pimentão
2712 Rami (fibra)
83380 Repolho
2713 Soja (em grão)
2714 Sorgo (em grão)
2715 Tomate
2716 Trigo (em grão)
109180 Triticale (em grão)


In [ ]:
for classificacao in metadados["classificacoes"]:
    print(classificacao["id"], classificacao["nome"])

81 Produto das lavouras temporárias


In [ ]:
id_milho = 2711
id_soja = 2713
id_cana = 2696
#Área plantada Hectares
area_plant_H = 109
#Área plantada - percentual do total geral %
area_plant_100 = 1000109
#Área colhida Hectares
area_colhida_H = 216
#Área colhida - percentual do total geral %
area_colhida_100 = 1000216
#Quantidade produzida Toneladas
quant_prod_T = 214
#Rendimento médio da produção Quilogramas por Hectare
rend_medio_Kg_H = 112
#Valor da produção Mil Cruzeiros [1974 a 1985, 1990 a 1992],
#  Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], 
# Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025]
valor_prod_moeda = 215
#Valor da produção - percentual do total geral %
valor_prod_100 = 1000215

In [ ]:
periodo = 2024
url_variaveis_2024_quant_prod = (
    f"https://servicodados.ibge.gov.br/api/v3/agregados/"
    f"{tabela_lavouras_temporárias}/periodos/{periodo}/variaveis/{quant_prod_T}"
)

parametros_sorriso_soja = {
    #N6 municipal e código de Sorriso
    "localidades": "N6[5107925]",
    #Código de lavouras temporarias e soja
    "classificacao": f"81[{id_soja}]"
}


with criar_sessao_ibge() as sessao:
    response = sessao.get(url_variaveis_2024_quant_prod,params=parametros_sorriso_soja ,timeout=30)
    response.raise_for_status()
    dados = response.json()

print("URL consultada:", response.url)
print(dados)
